# Prompt 6: Fault Tolerance & Resilience
## Databricks Certified Associate Developer for Apache Spark
### Topic 1 — Apache Spark Architecture & Components (20%)

---

Spark guarantees **fault tolerance** without replication — instead of copying data across nodes,
Spark records *how* each dataset was created and rebuilds it on demand.

**Exam facts to memorise:**
- RDD lineage = the DAG of transformations that produced a dataset — Spark's primary fault-tolerance mechanism
- `persist()` / `cache()` store computed data in memory (and/or disk) to avoid recomputation
- `checkpoint()` breaks lineage by saving to reliable storage (HDFS/S3) — used for long iterative lineages
- Storage levels: `MEMORY_ONLY`, `MEMORY_AND_DISK`, `DISK_ONLY`, `MEMORY_ONLY_SER`, `MEMORY_AND_DISK_SER`
- `cache()` = `persist(StorageLevel.MEMORY_ONLY)` for DataFrames
- Calling `cache()` / `persist()` is **lazy** — data is not cached until an action triggers computation
- Task failure: Spark retries the failed task up to `spark.task.maxFailures` (default **4**) times
- Stage failure: Spark reruns only the failed stage (reads shuffle map output if available)
- Executor failure: Spark releases the lost executor, recomputes its partitions using lineage

## 1. RDD Lineage — The Foundation of Fault Tolerance

Every RDD (and DataFrame, which is backed by an RDD) knows:
- Its **parent datasets** (the datasets it was derived from)
- The **transformation** applied to those parents to produce itself

This chain of parents + transformations is the **lineage graph** (or RDD DAG).
Spark never needs to replicate data across nodes — if a partition is lost, Spark traces back through
the lineage, locates the nearest available ancestor, and **recomputes** the lost partition.

### Lineage Example

```
raw_rdd  = sc.textFile('logs.txt')          # Stage 0: read from HDFS
    |
    | filter (narrow)
    v
errors   = raw_rdd.filter(lambda l: 'ERROR' in l)
    |
    | map (narrow)
    v
parsed   = errors.map(parse_log_line)
    |
    | groupByKey (wide — shuffle boundary)
    v
grouped  = parsed.groupByKey()              # Stage 1: shuffle
```

If a partition of `grouped` is lost after the shuffle:
- Spark reads the **shuffle map output** for that partition from the shuffle file (if still available on disk)
- If the shuffle file is also lost (executor died): Spark reruns the upstream `Stage 0` for just the affected partitions, regenerating the shuffle input

### Narrow vs Wide Dependencies (affects recovery cost)

| Dependency | Definition | Recovery cost |
|------------|-----------|---------------|
| **Narrow** | Each child partition depends on at most ONE parent partition (`filter`, `map`, `select`) | Low — recompute just the lost partition |
| **Wide** | Each child partition depends on MULTIPLE parent partitions (`groupBy`, `join`, `orderBy`) | Higher — may need to recompute an entire upstream shuffle stage |

### Viewing lineage with `toDebugString()`
```python
rdd = sc.textFile('data.txt').filter(lambda x: x).map(str.upper)
print(rdd.toDebugString().decode())  # Shows dependency chain
```

## 2. Persistence and Caching Strategies

Without caching, every action on a DataFrame **recomputes** the entire lineage from source.
Caching materialises the DataFrame in memory (or on disk) so subsequent actions use the cached copy.

### Storage Levels

| Storage Level | Memory | Disk | Serialised | Replicas | Use Case |
|---------------|--------|------|------------|---------|----------|
| `MEMORY_ONLY` | ✅ | ❌ | ❌ (Java objects) | 1 | Default for RDDs; fast reads; evicted if not enough memory |
| `MEMORY_AND_DISK` | ✅ | ✅ (overflow) | ❌ | 1 | DataFrames default via `cache()`; spills to disk if memory full |
| `DISK_ONLY` | ❌ | ✅ | ✅ | 1 | Very large datasets; slowest access |
| `MEMORY_ONLY_SER` | ✅ | ❌ | ✅ | 1 | More memory-efficient than MEMORY_ONLY; slower (deserialise on access) |
| `MEMORY_AND_DISK_SER` | ✅ | ✅ (overflow) | ✅ | 1 | Balanced for large datasets |
| `MEMORY_ONLY_2` | ✅ | ❌ | ❌ | 2 | Replicated; survives executor loss without recomputation |
| `OFF_HEAP` | External (Tungsten) | ❌ | ✅ | 1 | Avoids JVM GC; requires `spark.memory.offHeap.enabled=true` |

> **Exam note:** `cache()` on a DataFrame is equivalent to `persist(StorageLevel.MEMORY_AND_DISK)`.  
> `cache()` on an RDD is equivalent to `persist(StorageLevel.MEMORY_ONLY)`.

### DataFrame API
```python
df.cache()                               # MEMORY_AND_DISK (DataFrame default)
df.persist()                             # same as cache() for DataFrames
df.persist(StorageLevel.MEMORY_ONLY)    # explicit level
df.unpersist()                           # remove from cache
```

### RDD API
```python
rdd.cache()                              # MEMORY_ONLY (RDD default)
rdd.persist(StorageLevel.MEMORY_AND_DISK)
rdd.unpersist()
```

### CRITICAL: Caching is LAZY
Calling `df.cache()` registers the intent but does NOT cache any data.
Data is only cached when the first **action** (e.g., `count()`, `show()`) is triggered.
The second action reuses the cache.

## 3. Checkpointing

Checkpointing **breaks the lineage** by saving the physical data to a reliable distributed storage (HDFS, S3, DBFS).
After a checkpoint, Spark discards all upstream lineage — the checkpoint file becomes the new "source".

### When to checkpoint vs cache

| | `cache()` / `persist()` | `checkpoint()` |
|---|---|---|
| **Stores to** | Executor memory / local disk | Reliable external storage (HDFS/S3) |
| **Breaks lineage?** | ❌ No — lineage still tracked | ✅ Yes — lineage truncated at checkpoint |
| **Survives executor failure?** | Maybe (with `_2` levels) | Yes — stored on reliable storage |
| **Use when** | DataFrame is reused multiple times in the same job | Lineage is very long (iterative ML); streaming state |
| **Eager (immediate)?** | No — lazy | Yes — `checkpoint()` triggers an action immediately |
| **Requires checkpoint dir?** | No | Yes — `spark.sparkContext.setCheckpointDir('hdfs://...')` |

### Two types of checkpointing

**1. RDD / DataFrame checkpoint (reliable)**
```python
sc.setCheckpointDir('hdfs:///checkpoints/')  # must be set before checkpointing
rdd.checkpoint()   # writes to HDFS, breaks lineage
rdd.count()        # triggers the checkpoint write
```

**2. Local checkpoint (unreliable — faster but no lineage truncation guarantee)**
```python
df.localCheckpoint()  # writes to executor local disk; faster but lost on executor failure
```

> **Exam note:** Checkpointing is mandatory in **Structured Streaming** for maintaining state across restarts.
> Without a checkpoint directory, a stateful streaming query cannot resume from where it left off.

In [ ]:
# Cell 1: Setup — SparkSession and sample data
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum as spark_sum, avg
from pyspark.storagelevel import StorageLevel
import time

spark = SparkSession.builder \
    .appName('FaultToleranceDemo') \
    .master('local[*]') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

# Simulate a moderately expensive DataFrame: 500k rows with multi-step transformations
raw = spark.range(500_000).toDF('id')
transformed = (
    raw
    .withColumn('category', (col('id') % 5).cast('string'))   # 5 categories
    .withColumn('value', (col('id') * 1.5 + 10).cast('double'))
    .filter(col('id') % 2 == 0)   # keep even rows
)

print('DataFrame created (no data materialised yet — lazy evaluation)')
transformed.printSchema()

In [ ]:
# Cell 2: Caching demo — measure recomputation vs cache hit

# --- Without cache: each action recomputes from scratch ---
t0 = time.time()
cnt1 = transformed.count()         # full recomputation
t1 = time.time()
total1 = transformed.agg(spark_sum('value')).collect()[0][0]   # full recomputation again
t2 = time.time()
print(f'WITHOUT cache — count: {cnt1:,}  ({t1-t0:.2f}s)')
print(f'WITHOUT cache — sum:   {total1:,.0f}  ({t2-t1:.2f}s)')

# --- With cache: second action is much faster ---
transformed.cache()   # registers intent — data not cached yet

t3 = time.time()
cnt2 = transformed.count()         # triggers caching
t4 = time.time()
total2 = transformed.agg(spark_sum('value')).collect()[0][0]   # reads from cache
t5 = time.time()
print(f'WITH cache    — count: {cnt2:,}  ({t4-t3:.2f}s)  [first action: caches the data]')
print(f'WITH cache    — sum:   {total2:,.0f}  ({t5-t4:.2f}s)  [cache hit — much faster]')

# Release cache
transformed.unpersist()
print('\nCache released with unpersist()')

In [ ]:
# Cell 3: Explicit storage levels with persist()

# MEMORY_AND_DISK (DataFrame default via cache())
transformed.persist(StorageLevel.MEMORY_AND_DISK)
transformed.count()   # materialise cache
print('Persisted with MEMORY_AND_DISK')
print(f'Is cached: {transformed.is_cached}')   # True after materialisation
transformed.unpersist()
print(f'After unpersist: {transformed.is_cached}')  # False

print()

# MEMORY_ONLY_SER — serialised; more compact but slower on access
transformed.persist(StorageLevel.MEMORY_ONLY_SER)
transformed.count()
print('Persisted with MEMORY_ONLY_SER')
transformed.unpersist()

print()

# DISK_ONLY — useful when dataset is too large for memory
transformed.persist(StorageLevel.DISK_ONLY)
transformed.count()
print('Persisted with DISK_ONLY')
transformed.unpersist()

# Exam reminder table
print()
print('='*60)
print('EXAM QUICK REF: cache() vs persist() defaults')
print('  DataFrame.cache()  => MEMORY_AND_DISK')
print('  RDD.cache()        => MEMORY_ONLY')
print('  persist(level)     => explicit level control')
print('='*60)

In [ ]:
# Cell 4: RDD lineage — toDebugString() shows the DAG
sc = spark.sparkContext

# Build a multi-step RDD lineage
base_rdd    = sc.parallelize(range(1000), numSlices=4)          # ParallelCollectionRDD
doubled_rdd = base_rdd.map(lambda x: x * 2)                    # MappedRDD (narrow)
filtered_rdd = doubled_rdd.filter(lambda x: x % 6 == 0)        # FilteredRDD (narrow)
keyed_rdd   = filtered_rdd.map(lambda x: (x % 5, x))           # MappedRDD (narrow)
grouped_rdd = keyed_rdd.groupByKey()                            # ShuffledRDD (wide)

# toDebugString shows the full lineage in a readable DAG format
print('RDD Lineage (toDebugString):')
print(grouped_rdd.toDebugString().decode())

# Now cache at an intermediate point — lineage above cache is not recomputed
filtered_rdd.cache()
filtered_rdd.count()  # materialise
print()
print('After caching filtered_rdd, re-building grouped_rdd from cache point:')
# Only keyed_rdd and grouped_rdd need recomputation; filtered_rdd reads from cache
new_grouped = filtered_rdd.map(lambda x: (x % 3, x)).groupByKey()
print(f'Groups: {sorted(new_grouped.mapValues(list).collect())}')

filtered_rdd.unpersist()

In [ ]:
# Cell 5: Checkpointing — break lineage by saving to reliable storage
import tempfile, os

# Use a local temp dir to simulate HDFS checkpoint directory
checkpoint_dir = tempfile.mkdtemp(prefix='spark_checkpoint_')
sc.setCheckpointDir(checkpoint_dir)
print(f'Checkpoint directory: {checkpoint_dir}')

# Build a long lineage (simulates many ML iterations)
rdd = sc.parallelize(range(10_000), numSlices=4)
for i in range(20):   # 20 chained transformations — lineage becomes very long
    rdd = rdd.map(lambda x: x + 1)

print(f'Lineage depth before checkpoint:\n{rdd.toDebugString().decode()[:300]}...')
print(f'(truncated for readability)')

# Checkpoint: write to storage, truncate lineage
rdd.checkpoint()
rdd.count()   # triggers actual checkpoint write (eager for RDD checkpoint)

print()
print(f'Lineage after checkpoint (should be very short):')
print(rdd.toDebugString().decode())

# Verify checkpoint file exists
ckpt_files = [f for f in os.listdir(checkpoint_dir) if f]
print(f'Checkpoint dir contents (subdirs per RDD): {len(ckpt_files)} item(s)')

In [ ]:
# Cell 6: Recovery simulation — demonstrate Spark's task retry behaviour
# spark.task.maxFailures (default 4): Spark retries a task this many times before aborting the job

print('Task failure configuration:')
max_failures = spark.conf.get('spark.task.maxFailures', '4')
print(f'  spark.task.maxFailures = {max_failures}  (default: 4)')
print()

# Demonstrate a flaky UDF that fails on first attempt (simulating transient failure)
attempt_count = sc.accumulator(0)

def flaky_transform(x):
    """Fails on the very first call per partition, succeeds on retry."""
    attempt_count.add(1)
    # In a real scenario, this might be a network timeout, transient DB error, etc.
    # Here we just always succeed to avoid actually throwing (can't raise in accumulator demo)
    return x * 2

result = sc.parallelize(range(100), 4).map(flaky_transform).sum()
print(f'Result: {result}')
print(f'Total task calls (accumulator): {attempt_count.value}')
print()
print('Key recovery facts for the exam:')
print('  - Task failure: retried up to spark.task.maxFailures (default 4) times')
print('  - Stage failure: only failed stage re-executed (shuffle files reused where available)')
print('  - Executor failure: lost executor evicted; partitions recomputed from lineage')
print('  - Driver failure: entire application must restart (Driver is single point of failure)')

## 4. Caching Best Practices

### When to cache
- DataFrame is referenced by **2+ actions** in the same job
- Computation is expensive (many transformations, large joins, UDFs)
- Dataset fits (or mostly fits) in aggregate executor memory
- Iterative algorithms (ML training loops) that re-read the same data each iteration

### When NOT to cache
- DataFrame is used **only once** — caching adds overhead for no benefit
- Dataset is larger than available memory and DISK_ONLY would be needed — may be slower than recompute
- The data source is already fast to read (e.g., Parquet on local SSD with column pruning)

### Choosing a Storage Level

```
Does data fit in memory?
    YES → MEMORY_ONLY (RDD) or cache() / MEMORY_AND_DISK (DataFrame)
    NO  → Can you tolerate recomputation if evicted?
              YES → MEMORY_ONLY (eviction causes recompute)
              NO  → MEMORY_AND_DISK (spills to disk; no recompute)
Is memory very constrained?
    YES → MEMORY_ONLY_SER (compact but slower reads)
Is executor failure tolerance critical (no lineage recompute)?
    YES → MEMORY_ONLY_2 (2x replicated) or MEMORY_AND_DISK_2
```

### Unpersist vs Destroy (broadcast variable parallel)
- `df.unpersist()` — removes cached blocks; DataFrame is still usable (recomputed on next action)
- `df.unpersist(blocking=True)` — waits for removal to complete before returning

### Caching in Structured Streaming
- `cache()` / `persist()` on a streaming DataFrame raises an error — caching individual micro-batches is not supported
- Use **stateful operations** (mapGroupsWithState, aggregations with watermarks) instead
- Use **checkpoint directory** for streaming state and offset tracking

## 5. Node Failure Recovery — How Spark Responds

### Executor Failure
1. Cluster manager detects the dead executor and notifies the Driver
2. Driver marks all tasks that were running on that executor as **failed**
3. Failed tasks are re-submitted to remaining executors
4. If the failed executor held **shuffle map output**, the upstream stage is re-run to regenerate it
5. Any **cached partitions** on the failed executor are recomputed from lineage on demand

### Task Failure
- Spark retries the task on a **different executor** up to `spark.task.maxFailures` (default **4**) times
- After `maxFailures` retries, the stage fails and (unless `spark.stage.maxConsecutiveAttempts` allows retries) the job fails
- Speculative execution (`spark.speculation=true`): Spark launches a duplicate copy of a **slow** (not failed) task on another executor; whichever finishes first wins

### Driver Failure
- The Driver is a **single point of failure** — if it dies, the entire Spark application is lost
- In YARN `cluster` mode, YARN can restart the Driver (Application Master) automatically
- Structured Streaming recovers Driver restarts using the checkpoint directory (stores offsets + state)

### Stage Failure vs Job Failure

| Event | Spark Response |
|-------|---------------|
| Single task failure | Retry task on different executor (up to maxFailures) |
| Shuffle map output lost | Re-run upstream stage to regenerate shuffle data |
| Cached partition lost | Recompute from lineage (transparent to user) |
| All retries exhausted | Stage marked failed; job aborted |
| Driver dies (cluster mode) | Cluster manager may restart Driver; streaming recovers via checkpoint |

## 6. Real-World Scenarios

<details>
<summary>Scenario 1: ETL Pipeline — Caching a Filtered DataFrame Used by Multiple Downstream Aggregations</summary>

**Situation:**
A daily ETL job reads 50 GB of raw sales data, filters it to the current month, then runs 5 separate
aggregations (by region, product, salesperson, channel, day). Without caching, each aggregation
re-reads and re-filters the 50 GB source — 5 full scans.

**PySpark Code:**
```python
from pyspark.sql.functions import col, sum as spark_sum, avg

raw = spark.read.parquet('/data/sales/')           # 50 GB
this_month = raw.filter(col('sale_date').between('2026-04-01', '2026-04-30'))

# Cache after expensive filter — data materialised on first action
this_month.cache()

# 5 aggregations — each reads from cache, not re-reading 50 GB each time
by_region     = this_month.groupBy('region').agg(spark_sum('amount'))
by_product    = this_month.groupBy('product').agg(spark_sum('amount'))
by_salesperson= this_month.groupBy('salesperson').agg(avg('amount'))
by_channel    = this_month.groupBy('channel').agg(spark_sum('amount'))
by_day        = this_month.groupBy('sale_date').agg(spark_sum('amount'))

# Write all outputs
for df, name in [(by_region,'region'),(by_product,'product'),
                 (by_salesperson,'salesperson'),(by_channel,'channel'),(by_day,'day')]:
    df.write.parquet(f'/output/sales_by_{name}/', mode='overwrite')

this_month.unpersist()   # free memory after all aggregations complete
```

**Expected Outcome:**
50 GB source is read and filtered **once** (cache materialised on first write action).
Subsequent 4 aggregations read from cache. Total I/O: ~50 GB instead of ~250 GB.
Job runtime drops from ~45 min to ~12 min.

**Exam Sub-topic:** `cache()` / `persist()`; lazy caching; `unpersist()`; when to cache
</details>

<details>
<summary>Scenario 2: Machine Learning — Checkpointing to Truncate Iterative Lineage</summary>

**Situation:**
A team trains a custom gradient descent algorithm in PySpark with 100 iterations.
Each iteration applies transformations to a weights RDD. After 100 iterations the lineage graph
contains 100 chained transformation steps — when a task fails, Spark must trace all 100 steps
back to source to recompute, causing massive overhead and stack overflow in the DAGScheduler.

**PySpark Code:**
```python
sc.setCheckpointDir('hdfs:///ml_checkpoints/')

weights = sc.parallelize([0.0] * 100, numSlices=4)

for iteration in range(100):
    # Apply gradient update (expensive transformation)
    weights = weights.map(lambda w: w - 0.01 * gradient(w))

    # Checkpoint every 10 iterations to keep lineage short
    if (iteration + 1) % 10 == 0:
        weights.checkpoint()   # marks for checkpointing
        weights.count()        # triggers checkpoint write to HDFS
        print(f'Iteration {iteration+1}: checkpoint written')
        # After this point, lineage is truncated — recomputation only goes back 10 steps max

final_weights = weights.collect()
```

**Expected Outcome:**
Lineage depth is capped at ~10 steps (between checkpoints) instead of growing to 100.
Task failures recompute at most 10 steps back. Stack overflow in DAGScheduler is prevented.
Recovery time per failure drops from minutes to seconds.

**Exam Sub-topic:** `checkpoint()`; lineage truncation; `setCheckpointDir()`; iterative algorithm best practices
</details>

<details>
<summary>Scenario 3: Production Job — Executor Failure During a Join Stage</summary>

**Situation:**
A Spark job joins a 200 GB orders table with a 5 GB customers table on `customer_id`.
During the shuffle phase, one of the 10 executors crashes (hardware failure).
The shuffle map output on the dead executor is lost. Spark must recover automatically.

**How Spark Recovers (no code change needed — automatic):**
```
Timeline:
1. Executor 7 crashes mid-shuffle
2. Driver detects lost executor via heartbeat timeout
3. Driver marks all tasks on Executor 7 as FAILED
4. Tasks assigned to remaining 9 executors
5. DAGScheduler detects shuffle map output for Executor 7 is lost
6. DAGScheduler re-submits the upstream map stage for the partitions that were on Executor 7
7. Recovered shuffle map output written to surviving executors
8. Downstream join stage continues with recovered data
```

**Configuration for resilience:**
```python
# Allow job to proceed with fewer executors (dynamic allocation)
spark.conf.set('spark.dynamicAllocation.enabled', 'true')
spark.conf.set('spark.dynamicAllocation.minExecutors', '5')

# How many times to retry a task before failing the job
spark.conf.set('spark.task.maxFailures', '8')  # raise above default 4 for unstable clusters

# Speculative execution: launch duplicate of slow tasks
spark.conf.set('spark.speculation', 'true')
spark.conf.set('spark.speculation.multiplier', '1.5')  # task > 1.5x median = speculated
```

**Expected Outcome:**
Job completes successfully with a ~10–15% slowdown for the recovery.
No data loss. No intervention required from the engineer.

**Exam Sub-topic:** Executor failure recovery; lineage replay; `spark.task.maxFailures`; speculative execution
</details>

<details>
<summary>Scenario 4: Structured Streaming — Checkpoint Directory for Exactly-Once Recovery</summary>

**Situation:**
A streaming pipeline reads from Kafka, applies aggregations, and writes results to Delta Lake.
The Driver process is restarted during a maintenance window. Without a checkpoint directory,
the streaming query loses all state and re-reads from the beginning of the Kafka topic.

**PySpark Code:**
```python
# Checkpoint directory MUST be set for stateful streaming
CHECKPOINT_DIR = 'hdfs:///streaming/checkpoints/orders_agg/'

# Read from Kafka
raw_stream = (
    spark.readStream
    .format('kafka')
    .option('kafka.bootstrap.servers', 'broker:9092')
    .option('subscribe', 'orders')
    .load()
)

# Aggregation with watermark (stateful)
agg_stream = (
    raw_stream
    .selectExpr('CAST(value AS STRING) as json')
    .withWatermark('event_time', '10 minutes')
    .groupBy('region')
    .count()
)

# Checkpoint enables:
# 1. Kafka offset tracking (resume from last committed offset on restart)
# 2. State store recovery (re-loads aggregation state)
# 3. Exactly-once sink delivery (tracks which micro-batches were written)
query = (
    agg_stream.writeStream
    .format('delta')
    .option('checkpointLocation', CHECKPOINT_DIR)   # CRITICAL
    .outputMode('update')
    .start('/delta/orders_agg/')
)

query.awaitTermination()
```

**Expected Outcome:**
After Driver restart, the streaming query re-reads the checkpoint directory,
resumes from the last committed Kafka offset, and restores the aggregation state.
No duplicate writes. No lost data. Exactly-once semantics maintained.

**Exam Sub-topic:** `checkpointLocation`; streaming fault tolerance; offset tracking; stateful recovery
</details>

<details>
<summary>Scenario 5: Cost Optimisation — Choosing the Wrong Storage Level Causes OOM</summary>

**Situation:**
A team caches a 30 GB joined DataFrame with `MEMORY_ONLY` (the RDD default) on a cluster with
only 20 GB aggregate executor memory. Spark evicts the partitions that don't fit, but because
`MEMORY_ONLY` does not spill to disk, Spark silently **recomputes** evicted partitions on every access,
negating all caching benefits and causing repeated full recomputation.

**Wrong Code:**
```python
from pyspark.storagelevel import StorageLevel

joined = large_df.join(medium_df, on='id')  # 30 GB result
joined.persist(StorageLevel.MEMORY_ONLY)    # WRONG for 30 GB on 20 GB cluster
joined.count()  # only 20 GB cached; 10 GB evicted silently
joined.count()  # evicted partitions recomputed — no speedup, extra overhead
```

**Correct Code:**
```python
# Option 1: MEMORY_AND_DISK — spills overflow to disk; avoids silent recomputation
joined.persist(StorageLevel.MEMORY_AND_DISK)
joined.count()  # 20 GB in memory, 10 GB on disk — all accessible without recompute
joined.count()  # reads from memory + disk — consistent performance

# Option 2: MEMORY_AND_DISK_SER — serialised version uses ~3x less memory
joined.persist(StorageLevel.MEMORY_AND_DISK_SER)
# 30 GB uncompressed -> ~10 GB serialised — may now fit entirely in memory

# Option 3: Don't cache — if recompute is fast and memory is tight
# joined.unpersist()  and let Spark recompute; sometimes cheaper than disk spill
```

**Expected Outcome:**
`MEMORY_AND_DISK` stores what fits in memory and spills the rest to local executor disk.
All partitions are recoverable without re-reading the source. Performance is consistent.

**Exam Sub-topic:** Storage level selection; `MEMORY_ONLY` eviction behaviour; `MEMORY_AND_DISK`; serialised levels
</details>

## 7. Exam Cheat Sheet

### Key Facts

| Fact | Value / Detail |
|------|----------------|
| Primary fault-tolerance mechanism | **RDD lineage** (DAG of parent datasets + transformations) |
| `cache()` on DataFrame | Equivalent to `persist(MEMORY_AND_DISK)` |
| `cache()` on RDD | Equivalent to `persist(MEMORY_ONLY)` |
| Caching is lazy? | **Yes** — data cached on first action after `cache()` call |
| Release cache | `df.unpersist()` |
| `checkpoint()` breaks lineage? | **Yes** — saves to reliable storage; upstream DAG discarded |
| `checkpoint()` is eager? | **Yes** — triggers an immediate action (data written to storage) |
| `localCheckpoint()` | Writes to executor local disk; faster but not fault-tolerant |
| Task max retries | `spark.task.maxFailures` (default **4**) |
| Speculative execution | `spark.speculation=true` — duplicate slow tasks |
| Streaming checkpoint purpose | Offset tracking + state recovery + exactly-once semantics |
| MEMORY_ONLY eviction | Evicted partitions silently **recomputed** (not read from disk) |
| MEMORY_AND_DISK eviction | Evicted partitions **spilled to disk** (no recomputation needed) |
| Narrow dependency recovery | Recompute only the **lost partition** |
| Wide dependency recovery | May require recomputing the **entire upstream stage** |

---

## 8. Practice Q&A

<details>
<summary>Q1: What is RDD lineage and how does it provide fault tolerance?</summary>

**A:** RDD lineage is the **DAG (directed acyclic graph) of transformations** that produced an RDD/DataFrame.
Every RDD records its parent datasets and the transformation applied to produce it.
If a partition is lost (due to executor failure), Spark traces the lineage back to the nearest
available ancestor and **recomputes** the lost partition — no data replication needed.
This is more efficient than HDFS-style replication for most Spark workloads.
</details>

<details>
<summary>Q2: What is the difference between cache() and persist(), and what storage level does each use by default?</summary>

**A:**
- `cache()` is a shortcut for `persist()` with the **default storage level**
- For **DataFrames**: both `cache()` and `persist()` default to `MEMORY_AND_DISK`
- For **RDDs**: `cache()` defaults to `MEMORY_ONLY`; `persist()` also defaults to `MEMORY_ONLY` unless a level is specified
- `persist(StorageLevel.DISK_ONLY)` gives explicit control over storage placement
- Both are **lazy** — data is materialised on the first action after the call
</details>

<details>
<summary>Q3: When should you use checkpointing instead of (or in addition to) caching?</summary>

**A:** Use checkpointing when:
1. **Lineage is very long** (e.g., 50+ iterative transformations in ML algorithms) — truncates the lineage to prevent stack overflow and reduce recovery cost
2. **Executor failure resilience is critical** — checkpoint to reliable storage (HDFS/S3) survives executor failures; cached data on a failed executor must be recomputed
3. **Structured Streaming** — mandatory for stateful queries; stores offsets and state store for exactly-once recovery

Use **both** when: lineage is long AND the dataset is accessed multiple times (cache for speed, checkpoint for reliability).
</details>

<details>
<summary>Q4: What happens when a partition stored with MEMORY_ONLY is evicted due to memory pressure?</summary>

**A:** With `MEMORY_ONLY`, evicted partitions are **silently dropped** and will be **recomputed from lineage** the next time they are needed. There is no disk fallback.

This means if your dataset is larger than available memory, `MEMORY_ONLY` provides no guarantee that caching will actually help — Spark may recompute many partitions repeatedly.

Use `MEMORY_AND_DISK` when the dataset may not fit entirely in memory — evicted partitions spill to local executor disk and are read from there (no recomputation).
</details>

<details>
<summary>Q5: How does Spark recover from executor failure during a shuffle stage?</summary>

**A:** When an executor fails during a shuffle:
1. The Driver detects the failed executor via heartbeat timeout
2. All tasks that were running on the failed executor are marked **FAILED**
3. The Driver checks whether the **shuffle map output** (written by the map stage to the failed executor's local disk) is still accessible
4. If shuffle output is **lost** (executor is dead and its disk is gone): Spark re-runs the **upstream map stage** for only the affected partitions to regenerate the shuffle data
5. If shuffle output is **still accessible** (e.g., external shuffle service): the reduce stage reads from it directly — no re-run needed
6. Failed tasks are re-submitted to surviving executors

> **Pro tip:** Enabling the **external shuffle service** (`spark.shuffle.service.enabled=true`) stores shuffle map output on the NodeManager (YARN) or a separate daemon — this means the output survives executor eviction, eliminating the need to re-run map stages.
</details>